In [0]:
from pydeequ.check import Check, CheckLevel
from pydeequ.verification import VerificationSuite, VerificationResult
from pyspark.sql import functions as F
import datetime as dt

try:
    arrival_date = dbutils.widget.get("arrival_date")
except Exception:
    arrival_date = dt.date.today().stftime("%Y-%m-%d"))
try: 
    catalog = dbutils.widget.get("catalog")
except Exception:
    catalog = "travel_bookings"
try: 
    schema = dbutils.widget.get("schema")
except Exception:
    schema = "default"

# Load customer data from bronze layer
# Filter data based on arrival_date or current date

src = spark.table(f"{catalog}.bronze.customer_inc").where(F.col("business_date") == F.to_date(F.lit("arrival_date")))

# Define data quality checks
check = (Check(spark, CheckLevel.Error, "Customer Data Check") 
    .hasSize(lambda x: x > 0)
    .isComplete("customer_name")
    .isComplete("customer_address")
    .isComplete("email")
    )

# Run DQ checks
result = (VerificationSuite().onData(src).addCheck(check).run())
df = VerificationResult.CheckResultAsDataFrame(result)
display(df)

# Validate DQ results and raise exception if any checks failed
# Ensures data quality before proceeding to downstream processing
if result.status != "Success":
    raise ValueError("Customer DQ check failed")
print("Customer DQ check passed")

# Create operations schema and DQ results table for audit tracking
# Stores DQ check results with metadata for monitoring and reporting
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.ops")
spark.sql(f"""
          CREATE TABLE IF NOT EXISTS {catalog}.ops.dq_results(
              business_date DATE
              , dataset STRING
              , check_name STRING
              , status STRING
              , constraint STRING
              , message STRING
              , recorded_at TIMESTAMP
              ) USING DELTA
          )""")

# Transform and store DQ results with metadata for audit trail
# Includes business_date, dataset name, and timestamp for tracking
out = (df.withColumn("business_date", F.lit(arrival_date))
       .withColumn("dataset", F.lit("customer"))
       .withColumn("recorded_at", F.current_timestamp())
)

out.select("business_date","dataset","check","check_status","constraint","constraint_status","constraint_message","recorded_at")\
    .write.mode("append").option("mergeSchema", "true").saveAsTable(f"{catalog}.ops.dq_results")